# 🎬 Movie Recommendation Engine
### Content-Based Filtering using TF-IDF & Cosine Similarity

---

**Author:** Sireesha Ragipati  
**Dataset:** TMDB Movie Dataset (~4,693 movies)  
**Algorithm:** Content-Based Filtering  
**Tech Stack:** Python · Pandas · Scikit-learn · Difflib

---

## 📌 Project Overview

This project builds a **Content-Based Movie Recommendation System** that suggests similar movies based on genre similarity. Given a movie title as input, the system:

1. Vectorizes movie genres using **TF-IDF** (Term Frequency-Inverse Document Frequency)
2. Computes pairwise **Cosine Similarity** between all movies
3. Ranks and returns the **Top 10 most similar movies**

### How Content-Based Filtering Works

| Step | Process |
|------|----------|
| 1 | Represent each movie as a vector of genre features |
| 2 | Calculate similarity between the query movie and all others |
| 3 | Return movies with the highest similarity scores |

> **Why TF-IDF over simple one-hot encoding?**  
> TF-IDF gives lower weight to very common genres (like 'Drama') and higher weight to rare, distinctive genres — producing better recommendations.

---
## 📦 Step 1: Import Libraries

In [1]:
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

---
## 📂 Step 2: Load & Explore the Dataset

The dataset contains **4,693 movies** with the following features:
- `index` — unique movie identifier
- `genres` — space-separated genre labels (e.g., `"Action Adventure Sci-Fi"`)
- `title` — movie title

> **Dataset Source:** TMDB (The Movie Database) — a popular open-source movie metadata dataset.

In [2]:
# Load the dataset
df = pd.read_csv("movies _recomendation.csv")

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Dataset Shape: (4693, 3)
Columns: ['index', 'genres', 'title']


,index,genres,title
0,0,Action Adventure Fantasy Science Fiction,Avatar
1,1,Adventure Fantasy Action,Pirates of the Caribbean: At World's End
2,2,Action Adventure Crime,Spectre
3,3,Action Crime Drama Thriller,The Dark Knight Rises
4,4,Action Adventure Science Fiction,John Carter


In [3]:
# Basic dataset info
print("=" * 40)
print(f"Total Movies  : {df.shape[0]}")
print(f"Total Columns : {df.shape[1]}")
print("=" * 40)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

Total Movies  : 4693
Total Columns : 3

Column Names: ['index', 'genres', 'title']

Data Types:
index      int64
genres    object
title     object
dtype: object


---
## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [4]:
# Check for missing values
print("Missing Values per Column:")
print("-" * 30)
missing = df.isnull().sum()
print(missing)
print("-" * 30)
print(f"\n⚠️  '{missing.idxmax()}' column has {missing.max()} missing values")
print(f"   These will be replaced with empty strings to avoid errors.")

Missing Values per Column:
------------------------------
index      0
genres    27
title      0
dtype: int64
------------------------------

⚠️  'genres' column has 27 missing values
   These will be replaced with empty strings to avoid errors.


In [5]:
# Top genres analysis
all_genres = df['genres'].dropna().str.split().explode()
genre_counts = all_genres.value_counts().head(10)

print("🎭 Top 10 Most Common Genres in Dataset:")
print("-" * 35)
for genre, count in genre_counts.items():
    bar = '█' * (count // 100)
    print(f"  {genre:<20} {count:>4}  {bar}")

🎭 Top 10 Most Common Genres in Dataset:
-----------------------------------
  Drama                2235  ██████████████████████
  Comedy               1699  ████████████████
  Thriller             1235  ████████████
  Action               1109  ███████████
  Romance               876  ████████
  Adventure             769  ███████
  Crime                 681  ██████
  Fiction               519  █████
  Science               519  █████
  Horror                507  █████


---
## 🧹 Step 4: Data Preprocessing

### Handling Missing Values in `genres`

We have **3 strategies** for missing genre values:

| Strategy | Code | Pros | Cons |
|----------|------|------|------|
| **Drop rows** | `df.dropna(subset=['genres'])` | Clean data | Lose movie records |
| **Fill with empty string** ✅ | `df['genres'].fillna('')` | Keeps all movies | Zero similarity for missing |
| **Fill with placeholder** | `df['genres'].fillna('Unknown')` | Preserves row | Groups all missing together |

**We use Strategy 2** — replacing NaN with empty string. This ensures no movie is dropped, and movies with missing genres will simply have zero similarity with all others.

In [6]:
# Fill missing genre values with empty string
df['genres'] = df['genres'].fillna('')

# Ensure genres column is string type
df['genres'] = df['genres'].astype(str)

# Verify no more missing values
print(f"Missing values after cleaning: {df['genres'].isnull().sum()}")
print(f"✅ Data preprocessing complete! {len(df)} movies ready for modeling.")

Missing values after cleaning: 0
✅ Data preprocessing complete! 4693 movies ready for modeling.


---
## 🤖 Step 5: Feature Extraction — TF-IDF Vectorization

### What is TF-IDF?

**TF-IDF** (Term Frequency-Inverse Document Frequency) converts text into numerical vectors.

$$\text{TF-IDF}(t, d) = TF(t,d) \times IDF(t)$$

Where:
- **TF(t, d)** = How often term `t` appears in document `d`  
- **IDF(t)** = `log(N / df(t))` — penalizes terms that appear in many documents

**In our case:**
- Each **movie** = a document
- Each **genre word** = a term
- Common genres like `"Drama"` get lower weights
- Rare genres like `"Western"` get higher weights → better discrimination

In [7]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform genre text to feature vectors
feature_vectors = vectorizer.fit_transform(df['genres'])

print("📊 TF-IDF Feature Matrix Info:")
print("-" * 40)
print(f"  Matrix Shape  : {feature_vectors.shape}")
print(f"  → {feature_vectors.shape[0]} movies × {feature_vectors.shape[1]} unique genre terms")
print(f"  Matrix Type   : Sparse (efficient memory usage)")
print(f"  Total Features: {len(vectorizer.get_feature_names_out())} genre terms")
print(f"\n🎭 All Genre Features Learned:")
print(sorted(vectorizer.get_feature_names_out()))

📊 TF-IDF Feature Matrix Info:
----------------------------------------
  Matrix Shape  : (4693, 22)
  → 4693 movies × 22 unique genre terms
  Matrix Type   : Sparse (efficient memory usage)
  Total Features: 22 genre terms

🎭 All Genre Features Learned:
['action', 'adventure', 'animation', 'comedy', 'crime', 'documentary', 'drama', 'family', 'fantasy', 'fiction', 'foreign', 'history', 'horror', 'movie', 'music', 'mystery', 'romance', 'science', 'thriller', 'tv', 'war', 'western']


---
## 📐 Step 6: Compute Cosine Similarity

### What is Cosine Similarity?

Cosine Similarity measures the **angle between two vectors** — not their magnitude.

$$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

| Score | Meaning |
|-------|----------|
| **1.0** | Identical genre profiles |
| **0.5–0.9** | Highly similar genres |
| **0.1–0.5** | Partially similar |
| **0.0** | No genre overlap |

> **Why Cosine over Euclidean Distance?**  
> Cosine similarity is scale-invariant — a movie with genres `"Action Action Action"` and one with just `"Action"` would still have similarity = 1.0. This makes it robust for genre-based matching.

In [8]:
# Compute pairwise cosine similarity for all movies
similarity = cosine_similarity(feature_vectors)

print("📐 Cosine Similarity Matrix:")
print("-" * 40)
print(f"  Matrix Shape : {similarity.shape}")
print(f"  → {similarity.shape[0]} × {similarity.shape[1]} pairwise comparisons")
print(f"  Diagonal     : All 1.0 (a movie is 100% similar to itself)")
print(f"\nSample of similarity matrix (first 5×5):")
print(np.round(similarity[:5, :5], 3))

📐 Cosine Similarity Matrix:
----------------------------------------
  Matrix Shape : (4693, 4693)
  → 4693 × 4693 pairwise comparisons
  Diagonal     : All 1.0 (a movie is 100% similar to itself)

Sample of similarity matrix (first 5×5):
[[1.    0.746 0.43  0.183 0.862]
 [0.746 1.    0.576 0.245 0.466]
 [0.43  0.576 1.    0.64  0.499]
 [0.183 0.245 0.64  1.    0.212]
 [0.862 0.466 0.499 0.212 1.   ]]


---
## 🎯 Step 7: Build the Recommendation Function

### Design Decisions:

1. **Fuzzy Matching with `difflib`** — Handles typos. User types `"iron man"` → system finds `"Iron Man"`
2. **Skip self (rank 0)** — The query movie itself always has similarity = 1.0, so we skip index 0
3. **Top-N configurable** — Default is 10 recommendations, easily adjustable

In [9]:
def get_movie_recommendations(movie_name, top_n=10):
    """
    Returns top-N similar movies based on genre using TF-IDF + Cosine Similarity.
    
    Parameters:
    -----------
    movie_name : str
        Name of the movie to base recommendations on (supports typos)
    top_n : int
        Number of recommendations to return (default: 10)
    
    Returns:
    --------
    list of str : Recommended movie titles
    """
    
    list_of_all_titles = df['title'].tolist()
    
    # Step 1: Fuzzy match the input to handle typos
    close_matches = difflib.get_close_matches(movie_name, list_of_all_titles)
    
    if not close_matches:
        print(f"❌ No match found for '{movie_name}'. Please try a different title.")
        return []
    
    # Step 2: Take the best match
    close_match = close_matches[0]
    
    # Step 3: Find the movie's index in the dataframe
    index_of_movie = df[df.title == close_match]['index'].values[0]
    
    # Step 4: Get similarity scores for this movie vs all others
    similarity_score = list(enumerate(similarity[index_of_movie]))
    
    # Step 5: Sort by similarity (descending)
    sorted_movies = sorted(similarity_score, key=lambda x: x[1], reverse=True)
    
    # Step 6: Collect top-N recommendations (skip index 0 = itself)
    recommendations = []
    for movie in sorted_movies[1:top_n + 1]:  # skip the first (self)
        title = df[df.index == movie[0]]['title'].values[0]
        score = round(movie[1], 4)
        recommendations.append((title, score))
    
    # Display results
    print(f"\n🎬 You searched for: '{movie_name}'")
    print(f"🔍 Best match found: '{close_match}'")
    print(f"\n✨ Top {top_n} Movies Recommended for You:\n")
    print(f"{'Rank':<6} {'Movie Title':<45} {'Similarity Score'}")
    print("-" * 70)
    for i, (title, score) in enumerate(recommendations, 1):
        bar = '▓' * int(score * 20)
        print(f"  {i:<4} {title:<45} {score:.4f}  {bar}")
    
    return [r[0] for r in recommendations]

print("✅ Recommendation function defined successfully!")

✅ Recommendation function defined successfully!


---
## 🚀 Step 8: Run the Recommendation Engine

### Test Case 1: Iron Man

In [10]:
# Test with Iron Man
recommended = get_movie_recommendations("iron man", top_n=10)


🎬 You searched for: 'iron man'
🔍 Best match found: 'Iron Man'

✨ Top 10 Movies Recommended for You:

Rank   Movie Title                                   Similarity Score
----------------------------------------------------------------------
  1    Avengers: Age of Ultron                       1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  2    The Avengers                                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  3    Captain America: Civil War                    1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  4    Iron Man 3                                    1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  5    Transformers: Revenge of the Fallen           1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  6    Transformers: Age of Extinction               1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  7    TRON: Legacy                                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  8    Star Trek Into Darkness                       1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  9    Pacific Rim                                   1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  10   Transformers

### Test Case 2: Interstellar

In [11]:
# Test with Interstellar
recommended = get_movie_recommendations("interstellar", top_n=10)


🎬 You searched for: 'interstellar'
🔍 Best match found: 'Interstellar'

✨ Top 10 Movies Recommended for You:

Rank   Movie Title                                   Similarity Score
----------------------------------------------------------------------
  1    The Martian                                   1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  2    A.I. Artificial Intelligence                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  3    Midnight Special                              1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  4    20,000 Leagues Under the Sea                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  5    Silent Running                                1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  6    Allegiant                                     0.9505  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  7    Jurassic Park                                 0.9505  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  8    The 5th Wave                                  0.9505  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  9    Star Trek IV: The Voyage Home                 0.9505  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  10   Stargate

### Test Case 3: Typo Handling — 'avngers' → Avengers

In [12]:
# Test fuzzy matching with a typo
recommended = get_movie_recommendations("avngers age of ultron", top_n=5)


🎬 You searched for: 'avngers age of ultron'
🔍 Best match found: 'Avengers: Age of Ultron'

✨ Top 5 Movies Recommended for You:

Rank   Movie Title                                   Similarity Score
----------------------------------------------------------------------
  1    Avengers: Age of Ultron                       1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  2    The Avengers                                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  3    Captain America: Civil War                    1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  4    Iron Man 3                                    1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  5    Transformers: Revenge of the Fallen           1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


---
## 🎮 Step 9: Interactive Mode — User Input

In [13]:
# Interactive user input
print("=" * 55)
print("       🎬 MOVIE RECOMMENDATION ENGINE 🎬")
print("=" * 55)
movie_name = input("Enter your favourite movie name: ")
get_movie_recommendations(movie_name, top_n=10)

       🎬 MOVIE RECOMMENDATION ENGINE 🎬


Enter your favourite movie name:  iron man



🎬 You searched for: 'iron man'
🔍 Best match found: 'Iron Man'

✨ Top 10 Movies Recommended for You:

Rank   Movie Title                                   Similarity Score
----------------------------------------------------------------------
  1    Avengers: Age of Ultron                       1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  2    The Avengers                                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  3    Captain America: Civil War                    1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  4    Iron Man 3                                    1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  5    Transformers: Revenge of the Fallen           1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  6    Transformers: Age of Extinction               1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  7    TRON: Legacy                                  1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  8    Star Trek Into Darkness                       1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  9    Pacific Rim                                   1.0000  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  10   Transformers

['Avengers: Age of Ultron',
 'The Avengers',
 'Captain America: Civil War',
 'Iron Man 3',
 'Transformers: Revenge of the Fallen',
 'Transformers: Age of Extinction',
 'TRON: Legacy',
 'Star Trek Into Darkness',
 'Pacific Rim',
 'Transformers: Dark of the Moon']

---
## 📊 Step 10: Model Analysis & Evaluation

### Similarity Distribution for a Sample Movie

In [ ]:
# Analyze similarity distribution for Iron Man
iron_man_index = df[df['title'] == 'Iron Man']['index'].values[0]
scores = similarity[iron_man_index]

print("📈 Similarity Score Distribution for 'Iron Man':")
print("-" * 45)
print(f"  Total movies compared : {len(scores)}")
print(f"  Movies with score > 0.9 : {(scores > 0.9).sum()} (extremely similar)")
print(f"  Movies with score > 0.5 : {(scores > 0.5).sum()} (highly similar)")
print(f"  Movies with score > 0.1 : {(scores > 0.1).sum()} (somewhat similar)")
print(f"  Movies with score = 0.0 : {(scores == 0.0).sum()} (no genre overlap)")
print(f"\n  Mean similarity  : {scores.mean():.4f}")
print(f"  Max similarity   : {scores.max():.4f}  (itself = 1.0)")
print(f"  Min similarity   : {scores.min():.4f}")

---
## 🔮 Step 11: Limitations & Future Improvements

### Current Approach — Content-Based Filtering

| ✅ Strengths | ❌ Limitations |
|-------------|----------------|
| No user data required | Only uses genre — ignores cast, director, plot |
| Works for new movies instantly | Cannot discover cross-genre recommendations |
| Fast & interpretable | "Filter bubble" — always recommends same genre |
| No cold start problem | Cannot personalize per user taste |

### 🚀 Potential Enhancements

1. **Multi-feature similarity** — Include cast, director, keywords, and plot overview
2. **Collaborative Filtering** — Use user ratings to find "users like you also liked"
3. **Hybrid Model** — Combine content + collaborative for better accuracy
4. **Deep Learning** — Word2Vec / BERT embeddings for semantic plot similarity
5. **User Interface** — Build a Streamlit web app for interactive recommendations

---
## ✅ Summary

| Component | Details |
|-----------|----------|
| **Dataset** | 4,693 TMDB movies with genre labels |
| **Feature Engineering** | TF-IDF Vectorization on genre text |
| **Similarity Metric** | Cosine Similarity |
| **Input Handling** | Fuzzy string matching (difflib) for typo tolerance |
| **Output** | Top-N similar movies ranked by score |
| **Algorithm Type** | Content-Based Filtering |

---

### 🔗 Connect

- **LinkedIn:** [linkedin.com/in/sireesha-ragipati](https://www.linkedin.com/in/sireesha-ragipati-269a10244/)